# Agentic Search with Database Query Tool

This notebook walks you through an agentic search example with a general-purpose search tool to execute ES|QL queries against an Elasticsearch index. 

It improves the first implementation by adding Agent Skills for writing ES|QL queries.

![](../img/agent_search_for_context_engineering_overview.png)

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [1]:
import os                                                                                                                                                                                                          
from dotenv import load_dotenv

load_dotenv()

LITELLM_API_KEY=os.getenv("LITELLM_API_KEY")
LITELLM_API_BASE=os.getenv("LITELLM_API_BASE")

ES_URL=os.getenv("ELASTICSEARCH_URL")
ES_USERNAME=os.getenv("ELASTICSEARCH_USERNAME")
ES_PASSWORD=os.getenv("ELASTICSEARCH_PASSWORD")

## Set up agent with a database query tool

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    model="llm-gateway/gpt-5.4-nano",
    temperature=0.5,
)

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to an **Elasticsearch index** with the conference data.

In [3]:
SYSTEM_PROMPT = open("01_system_prompt_vanilla_agentic_search.md", 'r').read()

print(SYSTEM_PROMPT)

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Elasticsearch (`conference_schedule`)

The conference sessions are indexed in Elasticsearch. One document per session.

Basic Auth: ELASTICSEARCH_URL + ELASTICSEARCH_USERNAME + ELASTICSEARCH_PASSWORD

| Field | Description |
|--------------|------------|
| `text` | The string that was embedded: each session’s title plus description (blank line between them). It does not include day, time, room, or speakers. |
| `vector` | Dense embedding of `text`, used for similarity search. |
| `metadata` | Structured fields (see metadata description below) |


| Field | Descriptio

### Create tools

Tools let a model interact with external systems by calling functions you define. Tools can depend on runtime context and also interact with agent memory.
Notice below how the get_user_location tool uses runtime context:


[Create a retriever tool](https://docs.langchain.com/oss/python/langgraph/agentic-rag)

#### Setup embedding model and vector store

[Jina Embeddings](https://docs.langchain.com/oss/python/integrations/embeddings/jina)

In [4]:
#from langchain_community.embeddings import JinaEmbeddings

# Set up the embedding model
#embeddings = JinaEmbeddings(
#    jina_api_key=JINA_API_KEY, #
#    model_name="jina-embeddings-v5-text-small"
#)

In [5]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    hosts=ES_URL,
    basic_auth=(ES_USERNAME, ES_PASSWORD)
)

response = es_client.esql.query(
    query='''FROM conference_schedule 
    | WHERE metadata.speakers LIKE "*Leonie*"
    | KEEP metadata.speakers, metadata.title, text
    | LIMIT 2
    ''',
    format="csv",
)

from pprint import pprint
pprint(response.body)

('metadata.speakers,metadata.title,text\r\n'
 'Leonie Monigatti,Rebranding Retrieval: From RAG over Agentic Retrieval to '
 'Agent Memory,Rebranding Retrieval: From RAG over Agentic Retrieval to Agent '
 'Memory\r\n')


#### Create a retriever tool using the [`@tool` decorator](https://reference.langchain.com/python/langchain-core/tools/convert/tool)

Make sure you're model is at least somewhat capable of generating ES|QL queries from scratch (for example, in this notebook we had to switch from `gpt-5.4-nano` to `gpt-5.4-mini`)

In [6]:
from langchain.tools import tool

@tool()
def execute_esql_query(esql_query: str) -> str:
    """Execute an ES|QL query against the conference_schedule index in Elasticsearch.

    Args:
        esql_query: The ES|QL query to execute

    Returns:
        A string containing the information about the sessions.
    """
    try:
        response = es_client.esql.query(
            query=esql_query,
            format="csv",
        )
        return response.body 

    except Exception as e:
        return f"Error executing ES|QL query: {e}"

retriever_tool = execute_esql_query

Test the retriever tool:

In [7]:
esql_query = '''FROM conference_schedule 
    | WHERE metadata.speakers LIKE "*Leonie*"
    | KEEP metadata.speakers, metadata.title, text
    | LIMIT 2
    '''

print(retriever_tool.invoke({"esql_query": esql_query}))

metadata.speakers,metadata.title,text
Leonie Monigatti,Rebranding Retrieval: From RAG over Agentic Retrieval to Agent Memory,Rebranding Retrieval: From RAG over Agentic Retrieval to Agent Memory



### Create the agent

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[retriever_tool],
)

## Run the agent

### Agent uses tool to retrieve context

In [9]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "Which sessions are about context engineering?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
        stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions are about context engineering?
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_QjyMIjURxxDl92D8KWoeOmZu)
 Call ID: call_QjyMIjURxxDl92D8KWoeOmZu
  Args:
    esql_query: FROM conference_schedule | WHERE LOWER(metadata.title) LIKE '%context%' OR LOWER(text) LIKE '%context engineering%' OR LOWER(text) LIKE '%context%' AND (LOWER(text) LIKE '%engineering%' OR LOWER(text) LIKE '%engineer%') | LIMIT 50
================================= Tool Message =================================
Name: execute_esql_query

Error executing ES|QL query: BadRequestError(400, 'parsing_exception', "line 1:61: token recognition error at: '''")
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_F3S1w6GqiJqF4i0WsR6Ag5Tc)
 Call ID: call_F3S1w6GqiJqF4i0WsR6Ag5Tc
  Args:
    esql_quer

Problem:
LLM generates incorrect ES|QL queries.

Solution ideas:
- Use stronger model (e.g., `gpt-5.4-nano` to `gpt-5.4-mini`)
- Edit tool description (e.g., "Make sure to use double quotes for strings in the query." or "The wildcard character is an asterisk (*), not a percent sign (%).")
- Use an Agent Skill for writing ES|QL queries

## Adding a Skill tool

Based on https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant#3-build-skill-middleware

### 1. Define Skill

In [10]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str  # Unique identifier for the skill
    description: str  # 1-2 sentence description to show in system prompt
    content: str  # Full skill content with detailed instructions

SKILLS: list[Skill] = [
    {
        "name": "elasticsearch-esql",
        "description": """Execute ES|QL (Elasticsearch Query Language) queries, use when the user wants to
  query Elasticsearch data, analyze logs, aggregate metrics, explore data, or create
  charts and dashboards from ES|QL results.""",
        "content": """
# Elasticsearch ES|QL

Execute ES|QL queries against Elasticsearch.

## What is ES|QL?

ES|QL (Elasticsearch Query Language) is a piped query language for Elasticsearch. It is **NOT** the same as SQL.

ES|QL uses pipes (`|`) to chain commands:
`FROM index | WHERE condition | STATS aggregation BY field | SORT field | LIMIT n`

## Critical Syntax Rules

### String Literals Use Double Quotes Only

ES|QL uses **double quotes** for string literals — never single quotes. This is the most common source of
`token recognition error at: '` failures. SQL habits lead models to write `'value'` when ES|QL requires `"value"`.

### Comparison Operators

- `==` Equal
- `!=` Not equal
- `<`, `<=`, `>`, `>=` Comparison
- `IS NULL`, `IS NOT NULL` Null checks

### Logical Operators

- `AND` Logical AND
- `OR` Logical OR
- `NOT` Logical NOT

### Pattern Matching

- `LIKE` Wildcard pattern (`*` zero or more chars, `?` single char)
- `RLIKE` Regular expression
- `IN` Value in list

## Step-by-Step Generation Process

1. Identify the Data Source **Question:** What index or data should be queried?
2. Identify Filters **Question:** What conditions should narrow the results?
4. Determine Output Type**Question:** Does the user want raw data or aggregated results?
5. Select Fields **Question:** What fields should be shown? For raw data queries, use KEEP to select relevant fields:
6. Apply Ordering and Limits: **Question:** How should results be ordered and limited?
7. **Always add LIMIT** unless the user specifically wants all results:

## When to Use Full-Text Search

Use full-text search functions (`MATCH`, `QSTR`, `KQL`, `MATCH_PHRASE`) when:

- Searching natural-language text (log messages, descriptions, titles, comments)
- Relevance ranking matters (most relevant results first)
- You need analyzer features: case-insensitive matching, stemming, synonyms, fuzzy matching
- Searching multivalued text fields
- Performance matters on large datasets

## Semantic Search

Search `semantic_text` fields using `MATCH` or `:` for vector-based semantic matching. No separate function is needed —
MATCH automatically performs semantic search on `semantic_text` fields.

```esql
// Semantic search via colon operator
FROM knowledge_base METADATA _score
| WHERE semantic_content : "how do I reset my password"
| SORT _score DESC
| LIMIT 10

// Semantic search via MATCH
FROM knowledge_base METADATA _score
| WHERE MATCH(semantic_content, "configure two-factor authentication")
| SORT _score DESC
| LIMIT 10
```

"""
    },
]

### 2. Create skill loading tool

Create a tool to load full skill content on-demand:

In [11]:
from langchain.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """Load the full content of a skill into the agent's context.

    Use this when you need detailed information about how to handle a specific
    type of request. This will provide you with comprehensive instructions,
    policies, and guidelines for the skill area.

    Args:
        skill_name: The name of the skill to load (e.g., "expense_reporting", "travel_booking")
    """
    # Find and return the requested skill
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"

    # Skill not found
    available = ", ".join(s["name"] for s in SKILLS)
    return f"Skill '{skill_name}' not found. Available skills: {available}"

### 3. Build skill middleware

Create custom middleware that injects skill descriptions into the system prompt. This middleware makes skills discoverable without loading their full content upfront.

In [12]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    # Register the load_skill tool as a class variable
    tools = [load_skill]

    def __init__(self):
        """Initialize and generate the skills prompt from SKILLS."""
        # Build skills prompt from the SKILLS list
        skills_list = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**: {skill['description']}"
            )
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Sync: Inject skill descriptions into system prompt."""
        # Build the skills addendum
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the load_skill tool when you need detailed information "
            "about handling a specific type of request."
        )

        # Append to system message content blocks
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

In [13]:
#from langchain_skills import SkillTool
from langchain.tools import tool

# Create Skill tool
#skill_tool = SkillTool(directories=["../.agents/skills"])

@tool()
def execute_esql_query(esql_query: str) -> str:
    """Execute an ES|QL query against and index in Elasticsearch.
    Always use the Elasticsearch ES|QL skill to generate the ES|QL query before using this tool to execute the query.

    Args:
        esql_query: The ES|QL query to execute

    Returns:
        A string containing the information about the sessions.
    """
    try:
        response = es_client.esql.query(
            query=esql_query,
            format="csv",
        )
        return response.body 

    except Exception as e:
        return f"Error executing ES|QL query: {e}"

retriever_tool = execute_esql_query

In [14]:
SYSTEM_PROMPT = SYSTEM_PROMPT +  "If you need to execute an ES|QL query, use the Elasticsearch ES|QL skill to generate the query before using this tool to execute the query. If an ESQL query returns an error use the Elasticsearch ES|QL skill to generate a new query."

In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    middleware=[SkillMiddleware()],
    tools=[retriever_tool],
)


In [16]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "What session are related to information retrieval?"#is most similar to retrieval?"#is Leonie's session about?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
        stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What session are related to information retrieval?
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_LBGb9HDxOkBgSeyYgsMQq9rP)
 Call ID: call_LBGb9HDxOkBgSeyYgsMQq9rP
  Args:
    skill_name: conference_schedule
================================= Tool Message =================================
Name: load_skill

Skill 'conference_schedule' not found. Available skills: elasticsearch-esql
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_xrJNhRlW6mZOefT0f8uWxQfY)
 Call ID: call_xrJNhRlW6mZOefT0f8uWxQfY
  Args:
    esql_query: FROM conference_schedule
| WHERE LOWER(metadata.title) LIKE "%retrieval%" OR LOWER(text) LIKE "%retrieval%" OR LOWER(metadata.title) LIKE "%information retrieval%" OR LOWER(text) LIKE "%information retrieval%" OR LOWER(metadata.title) LIKE "%retrieving%" OR LO

In [17]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "How many sessions are on the first day?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How many sessions are on the first day?
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_VopfcOrmHKaI17hmP5o25op3)
 Call ID: call_VopfcOrmHKaI17hmP5o25op3
  Args:
    esql_query: FROM conference_schedule | STATS count() AS sessions BY metadata.day | SORT metadata.day
================================= Tool Message =================================
Name: execute_esql_query

Error executing ES|QL query: BadRequestError(400, 'parsing_exception', "line 1:42: mismatched input 'AS' expecting {<EOF>, 'where', '|', 'and', 'by', '::', ',', 'or', '+', '-', '*', '/', '%'}")
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_34tG2LPoekG8Ujexj4u3CzR0)
 Call ID: call_34tG2LPoekG8Ujexj4u3CzR0
  Args:
    esql_query: FROM conference_schedule | STATS count() BY metadata.day | SORT meta

/var/folders/64/w4vvrnpx0nxf4rnr04bc39lc0000gn/T/ipykernel_55967/3451145287.py:19: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  response = es_client.esql.query(


================================== Ai Message ==================================

There are **21 sessions on the first day (April 8)**.


In [18]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "How many different tracks are there and which one are they?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How many different tracks are there and which one are they?
================================== Ai Message ==================================
Tool Calls:
  execute_esql_query (call_gEidGf9AS9FCpdgYR5ZvdPc0)
 Call ID: call_gEidGf9AS9FCpdgYR5ZvdPc0
  Args:
    esql_query: FROM conference_schedule | STATS track = count(metadata.track) BY metadata.track
================================= Tool Message =================================
Name: execute_esql_query

track,metadata.track
40,
8,AI Architects
7,Claws & Personal Agents
8,Coding Agents
8,Context Engineering
7,Evals & Observability
16,Expo Sessions (Shelley)
15,Expo Sessions (Westley)
16,Expo Sessions (Wordsworth)
1,GPUs & LLM Infra
7,GPUs & LLM Infrastructure
4,Generative Media
9,Harness Engineering
1,Leadership Lunch
8,MCP
7,Voice & Vision



/var/folders/64/w4vvrnpx0nxf4rnr04bc39lc0000gn/T/ipykernel_55967/3451145287.py:19: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  response = es_client.esql.query(


================================== Ai Message ==================================

There are **15 different tracks**:

1. **AI Architects**
2. **Claws & Personal Agents**
3. **Coding Agents**
4. **Context Engineering**
5. **Evals & Observability**
6. **Expo Sessions (Shelley)**
7. **Expo Sessions (Westley)**
8. **Expo Sessions (Wordsworth)**
9. **GPUs & LLM Infra**
10. **GPUs & LLM Infrastructure**
11. **Generative Media**
12. **Harness Engineering**
13. **Leadership Lunch**
14. **MCP**
15. **Voice & Vision**
